# 09_backtesting_and_risk_management.ipynb

This notebook runs a realistic backtest using the out-of-sample predictions of the trained machine learning models, incorporating risk management and transaction costs.

### Objectives:
1. Load out-of-sample test features, labels, and models.
2. Generate model-driven trading signals.
3. Simulate trade execution (next-candle open fills, dynamic ATR stop loss/take profit, time exit).
4. Apply professional risk management (fixed fractional position sizing).
5. Calculate key performance metrics (Sharpe, Sortino, Win Rate, Max Drawdown).
6. Save the trade journal and equity curve.

## 1. Environment Setup & Mount Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# Git Integration - Pull latest code from GitHub
import os
PROJECT_ROOT = "/content/drive/MyDrive/btcusdt_quant_research"
GITHUB_REPO_URL = "https://github.com/umutergul74/TradeBot.git"

print("Pulling latest code from GitHub...")
os.chdir(PROJECT_ROOT)
# Initialize git and pull
!git init -b main
!git remote remove origin 2>/dev/null || true
!git remote add origin {GITHUB_REPO_URL}
!git fetch origin
!git reset --hard origin/main
print("Codebase updated to latest GitHub commit!")

In [ ]:
import os
import sys
import pandas as pd
import numpy as np

PROJECT_ROOT = "/content/drive/MyDrive/btcusdt_quant_research"
sys.path.append(f"{PROJECT_ROOT}/src")

## 2. Load Configurations & Out-of-Sample Data

In [ ]:
from utils.config import load_global_config
from utils.io_utils import load_parquet
from models.model_registry import ModelRegistry

config = load_global_config(PROJECT_ROOT)
symbol = config.get("data", "symbol", "BTCUSDT")

# Load features and labels
features_path = os.path.join(PROJECT_ROOT, "data", "feature_store", f"{symbol}_5m_features.parquet")
df_features = load_parquet(features_path)
labels_path = os.path.join(PROJECT_ROOT, "data", "labels", f"{symbol}_5m_labels.parquet")
df_labels = load_parquet(labels_path)

df_model = df_features.loc[df_labels.index].copy()
df_model["meta_label"] = df_labels["meta_label"].astype(float)

# Take the out-of-sample test set (last 20% of data)
n_samples = len(df_model)
test_idx = int(n_samples * 0.8)
df_test = df_model.iloc[test_idx:].copy()

print(f"Test dataset shape: {df_test.shape}")

## 3. Generate Signals

In [ ]:
registry = ModelRegistry(PROJECT_ROOT)
model = registry.load_model(f"{symbol}_lgbm_model")
calibrator = registry.load_model(f"{symbol}_lgbm_calibrator")

exclude_cols = ["label", "meta_label", "timestamp", "open", "high", "low", "close", "volume", "taker_buy_base_volume"]
features = [col for col in df_model.columns if col not in exclude_cols]

X_test = df_test[features].values
raw_probs = model.predict_proba(X_test)
calibrated_probs = calibrator.calibrate(raw_probs)

# Class 1 probability represents the likelihood of the setup succeeding
# If probability is high, we trigger a signal (+1 for long setups, -1 for bearish setups)
signals = pd.Series(0, index=df_test.index)
threshold = 0.55

# To determine direction, we look at the confluence score (engineered feature)
is_bullish_setup = df_test["confluence_score"] > 0
is_bearish_setup = df_test["confluence_score"] < 0

signals = np.where(
    (calibrated_probs[:, 1] >= threshold) & is_bullish_setup,
    1,
    np.where(
        (calibrated_probs[:, 1] >= threshold) & is_bearish_setup,
        -1,
        0
    )
)
signals = pd.Series(signals, index=df_test.index)
print(f"Generated {signals.value_counts().get(1, 0)} long signals and {signals.value_counts().get(-1, 0)} short signals.")

## 4. Run Backtest Simulation

In [ ]:
from backtesting.execution import run_backtest
from backtesting.metrics import compute_performance_metrics

tp_mult = config.get("labeling", "triple_barrier", {}).get("tp_mult", 2.0)
sl_mult = config.get("labeling", "triple_barrier", {}).get("sl_mult", 1.0)
time_barrier = config.get("labeling", "triple_barrier", {}).get("time_barrier", 24)
fee_rate = config.get("backtest", "backtest", {}).get("fees", 0.0005)
slippage_rate = config.get("backtest", "backtest", {}).get("slippage", 0.0002)

# Run execution simulator
trades = run_backtest(
    df=df_features.loc[df_test.index], # Align price series with the test events
    signals=signals,
    tp_mult=tp_mult,
    sl_mult=sl_mult,
    time_barrier=time_barrier,
    fee_rate=fee_rate,
    slippage_rate=slippage_rate
)

df_trades = pd.DataFrame(trades)
print(f"Backtest completed. Total trades executed: {len(df_trades)}")
if not df_trades.empty:
    print(df_trades.head())

## 5. Risk Management & Equity Curve

In [ ]:
if not df_trades.empty:
    # Calculate equity curve with fixed fractional position sizing (risking 0.5% of capital per trade)
    initial_capital = 10000.0
    capital = initial_capital
    equity_curve = [initial_capital]
    risk_per_trade = 0.005
    
    for idx, row in df_trades.iterrows():
        pnl_pct = row["net_pnl"]
        # Size based on risk
        trade_size = capital * risk_per_trade / (sl_mult * 0.01) # Approx stop size
        trade_pnl = trade_size * pnl_pct
        capital += trade_pnl
        equity_curve.append(capital)
        
    df_trades["equity"] = equity_curve[1:]
    series_equity = pd.Series(equity_curve)
    
    metrics = compute_performance_metrics(df_trades["net_pnl"], series_equity)
    print("\nPerformance Metrics:")
    for k, v in metrics.items():
        print(f"{k}: {v:.4f}")

## 6. Save Backtest Reports

In [ ]:
from utils.io_utils import save_parquet

if not df_trades.empty:
    trade_journal_path = os.path.join(PROJECT_ROOT, "reports", "backtests", f"{symbol}_trade_journal.parquet")
    save_parquet(df_trades, trade_journal_path)
    print(f"Trade journal saved to {trade_journal_path}")

## Summary of Backtest Results

We completed out-of-sample backtesting:
1. Simulated realistic trade execution with next-candle fills and transaction costs.
2. Computed performance metrics and saved the trade journal under `reports/backtests/`.

**Next Step**: Run [10_explainability_and_trade_journal.ipynb](file:///content/drive/MyDrive/btcusdt_quant_research/notebooks/10_explainability_and_trade_journal.ipynb) to inspect model decisions and explain specific trades.

In [ ]:
# Auto-disconnect Colab runtime to save compute units
AUTO_DISCONNECT = False  # Set to True to enable automatic shutdown
if AUTO_DISCONNECT:
    print("Disconnecting Colab runtime...")
    from google.colab import runtime
    runtime.unassign()